# Clase 2 · El ICFES

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo durante la primera mitad de la clase. Ustedes lo siguen, lo corren y me interrumpen. En la segunda mitad cada uno resuelve el gemelo, *Dengue*, con la misma técnica y otros datos.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Limpiar un dato de personas dejando cada decisión a la vista, reconocer los sesgos que trae y los que agrega la limpieza, y escoger una métrica que responda a la decisión del cliente. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular el problema, los objetivos y la métrica), 2 (seleccionar y preparar datos), 4 (implementar y evaluar un modelo), 5 (comunicar a quien no es técnico) y 6 (reflexión ética, técnica y operativa). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea de código, que proponga cómo codificar una categoría o que escriba una gráfica. Deben poder explicar sin ayuda por qué la exactitud engaña en este problema, qué es la captación, por qué borrar filas con faltantes sesga la muestra, qué columnas filtran la respuesta y por qué la partición se hace por colegio. |
| **Entregable** | Este cuaderno corrido de principio a fin en su portafolio, con las predicciones escritas, la decisión en tres frases y la tabla de la sección 4.5 llenas. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

## 1. Formulación

Imagínense que la Secretaría de Educación de Caldas tiene un programa de refuerzo para estudiantes de grado once. El presupuesto alcanza para uno de cada cuatro. Los cupos se reparten meses antes del examen, cuando lo único que se sabe de cada estudiante es lo que respondió en el formulario de inscripción y en qué colegio estudia.

La Secretaría necesita tres respuestas:

1. **Predecir.** ¿Qué estudiantes van a quedar en el cuarto más bajo del país en Saber 11?
2. **Explicar.** ¿Qué condiciones del hogar y del colegio acompañan ese resultado?
3. **Decidir.** Con cupo para el 25 %, ¿a quiénes se les ofrece el refuerzo, y a quiénes deja por fuera esa regla?

**El dato.** El ICFES publica los resultados anonimizados de cada estudiante que presentó Saber 11. Usamos la aplicación 2020-2, que trae 504.872 estudiantes del país. Cada fila es una persona, con su puntaje, las respuestas del cuestionario socioeconómico y los datos del colegio. Nosotros bajamos un departamento.

**La respuesta que vamos a modelar.** El ICFES entrega con cada puntaje global su percentil nacional. Llamo $y_i=1$ al estudiante $i$ cuyo percentil es 25 o menos, y $y_i=0$ a los demás. El corte en 25 es una decisión mía, y por eso va a la bitácora.

**La métrica.** La Secretaría no quiere un porcentaje de aciertos. Quiere que sus cupos lleguen a quienes iban a quedar abajo. Si el modelo ordena a los $n$ estudiantes por riesgo y el programa atiende a los $k=\lfloor 0{,}25\,n\rceil$ primeros, la métrica es la **captación**:

$$\text{captación}=\frac{\#\{i:\ y_i=1\ \text{y}\ i\ \text{recibe cupo}\}}{\#\{i:\ y_i=1\}}.$$

Una captación de 0,60 significa que, de cada diez estudiantes que quedaron en el cuarto más bajo, seis tenían cupo.

### Predicción escrita

Antes de ver un solo dato, escriban aquí su apuesta. Después la comparamos con lo que salga.

- En Caldas, ¿qué fracción de los estudiantes queda en el cuarto más bajo del país? **Mi apuesta:**
- Si los cupos se repartieran por sorteo, ¿cuál sería la captación? **Mi apuesta:**
- Con lo que se sabe en la inscripción, ¿qué captación logrará el mejor modelo? **Mi apuesta:**
- ¿Qué variable pesará más: el estrato, la educación de la madre, los libros que hay en la casa o el tipo de colegio? **Mi apuesta:**

## 2. Datos

La primera celda fija los dos parámetros del cuaderno. La segunda trae el dato desde datos.gov.co y tarda unos veinte segundos.

In [ ]:
DEPARTAMENTO = "CALDAS"      # al final de la clase lo cambiarán por otro departamento
CUPO = 0.25                  # fracción de estudiantes para la que alcanza el programa

In [ ]:
crudo = descargar("saber11", departamento=DEPARTAMENTO)
print("Filas:", len(crudo), "| Columnas:", crudo.shape[1], "| Colegios:", crudo["cole_cod_dane_establecimiento"].nunique())
print("¿Algún estudiante repetido?", crudo["estu_consecutivo"].duplicated().any())
crudo.head(3).T

### 2.1 Las trampas de la limpieza

Limpiar es decidir. Cada regla de limpieza cambia quién queda en la tabla, y con eso cambia la respuesta. Voy a mostrar cinco trampas de este dato y la decisión que tomo en cada una.

**Primera trampa: fechas de nacimiento imposibles.** La edad no viene en la tabla. Hay que calcularla con la fecha de nacimiento, y esa fecha la digitó una persona.

In [ ]:
anio_nacimiento = pd.to_numeric(crudo["estu_fechanacimiento"].str[:4], errors="coerce")
edad = 2020 - anio_nacimiento
print("Edades más frecuentes:"); print(edad.value_counts().head(5).sort_index().to_string())
print("\nEdades que no pueden ser de un estudiante de once:")
print(edad[(edad < 13) | (edad > 70)].value_counts().sort_index().to_string())

Nadie presenta Saber 11 a los cero años. Esas fechas son errores de digitación. Si las dejo, la variable edad tendrá valores absurdos. Si borro las filas, pierdo estudiantes que sí presentaron el examen. **Decisión:** la edad por fuera del rango de 13 a 70 años se declara faltante y la fila se conserva.

**Segunda trampa: puntajes que no miden nada.**

In [ ]:
areas = ["punt_lectura_critica", "punt_matematicas", "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles", "punt_global"]
print("Estudiantes con puntaje global igual a cero:", int((crudo["punt_global"] == 0).sum()))
display(crudo.loc[crudo["punt_global"] == 0, areas + ["percentil_global", "estu_estadoinvestigacion"]])
print(crudo["estu_estadoinvestigacion"].value_counts().to_string())

Un cero en las cinco áreas no describe a un estudiante que respondió mal. Describe una prueba que no se respondió o que no se calificó. La tabla no dice cuál de las dos. **Decisión:** esas filas salen, porque no hay un resultado que predecir. Es la única regla de este cuaderno que borra filas.

**Tercera trampa: categorías que parecen respuestas.** Miren el estrato y la educación de la madre.

In [ ]:
for columna in ["fami_estratovivienda", "fami_educacionmadre"]:
    print(crudo[columna].value_counts(dropna=False).to_string(), "\n")

Hay tres clases de valor que conviene distinguir. Están las respuestas con orden, como «Estrato 1» o «Primaria completa». Está la celda vacía, que es un estudiante que no respondió. Y están «Sin Estrato», «No sabe» y «No Aplica», que sí son respuestas pero no caben en el orden. Si convierto «Sin Estrato» en un cero, le digo al modelo que es un estrato más bajo que el uno, y eso no lo dice el dato.

**Decisión:** las categorías con orden se vuelven números. «No sabe» y «No Aplica» se tratan como faltantes. «Sin Estrato» queda con su propia columna, porque es información distinta.

In [ ]:
niveles_educacion = {
    "Ninguno": 0, "Primaria incompleta": 1, "Primaria completa": 2,
    "Secundaria (Bachillerato) incompleta": 3, "Secundaria (Bachillerato) completa": 4,
    "Técnica o tecnológica incompleta": 5, "Técnica o tecnológica completa": 6,
    "Educación profesional incompleta": 7, "Educación profesional completa": 8, "Postgrado": 9,
}
libros = {"0 A 10 LIBROS": 0, "11 A 25 LIBROS": 1, "26 A 100 LIBROS": 2, "MÁS DE 100 LIBROS": 3}
personas = {"1 a 2": 1.5, "3 a 4": 3.5, "5 a 6": 5.5, "7 a 8": 7.5, "9 o más": 9}
cuartos = {"Uno": 1, "Dos": 2, "Tres": 3, "Cuatro": 4, "Cinco": 5, "Seis o mas": 6}
lectura = {"No leo por entretenimiento": 0, "30 minutos o menos": 1, "Entre 30 y 60 minutos": 2, "Entre 1 y 2 horas": 3, "Más de 2 horas": 4}
trabajo = {"0": 0, "Menos de 10 horas": 1, "Entre 11 y 20 horas": 2, "Entre 21 y 30 horas": 3, "Más de 30 horas": 4}

X = pd.DataFrame(index=crudo.index)
X["mujer"] = (crudo["estu_genero"] == "F").astype(float).where(crudo["estu_genero"].notna())
X["edad"] = edad.where(edad.between(13, 70))
X["etnia"] = (crudo["estu_tieneetnia"] == "Si").astype(float).where(crudo["estu_tieneetnia"].notna())
X["estrato"] = pd.to_numeric(crudo["fami_estratovivienda"].str.extract(r"(\d)")[0], errors="coerce")
X["sin_estrato"] = (crudo["fami_estratovivienda"] == "Sin Estrato").astype(float)
X["educacion_madre"] = crudo["fami_educacionmadre"].map(niveles_educacion)
X["educacion_padre"] = crudo["fami_educacionpadre"].map(niveles_educacion)
for cosa in ["internet", "computador", "automovil"]:
    X[cosa] = (crudo[f"fami_tiene{cosa}"] == "Si").astype(float).where(crudo[f"fami_tiene{cosa}"].notna())
X["libros"] = crudo["fami_numlibros"].map(libros)
X["personas_hogar"] = crudo["fami_personashogar"].map(personas)
X["cuartos"] = crudo["fami_cuartoshogar"].map(cuartos)
X["lectura_diaria"] = crudo["estu_dedicacionlecturadiaria"].map(lectura)
X["horas_trabajo"] = crudo["estu_horassemanatrabaja"].astype(str).map(trabajo)
X["colegio_oficial"] = (crudo["cole_naturaleza"] == "OFICIAL").astype(float)
X["colegio_rural"] = (crudo["cole_area_ubicacion"] == "RURAL").astype(float)
X["nocturna_o_sabatina"] = crudo["cole_jornada"].isin(["NOCHE", "SABATINA"]).astype(float)
X["jornada_completa"] = crudo["cole_jornada"].isin(["COMPLETA", "UNICA"]).astype(float)

y = (crudo["percentil_global"] <= 25).astype(int).rename("bajo")
colegio = crudo["cole_cod_dane_establecimiento"]

validas = crudo["punt_global"] > 0                     # la única regla que borra filas
X, y, colegio, datos = X[validas], y[validas], colegio[validas], crudo[validas]
variables = list(X.columns)
print(f"Quedan {len(X)} estudiantes y {len(variables)} variables.")
print(f"Quedan en el cuarto más bajo del país: {y.mean() * 100:.1f} %")

Comparen con su primera apuesta. Si el departamento fuera una copia del país, ese número sería 25 %.

**Cuarta trampa: los faltantes no caen al azar.** La salida fácil para las celdas vacías es `dropna()`, que borra toda fila con algún faltante. Antes de usarla hay que preguntarse quiénes son los que no respondieron. Llamo $m_i=1$ al estudiante con alguna variable faltante. Borrar filas es inofensivo solo si la respuesta no depende de eso:

$$P(y=1\mid m=1)=P(y=1\mid m=0).$$

**Predicción escrita:** ¿los estudiantes con respuestas faltantes quedan abajo con la misma frecuencia que los demás? **Mi apuesta:**

In [ ]:
faltantes = (X.isna().mean() * 100).sort_values()
faltantes[faltantes > 0].plot.barh(color="#16302B", figsize=(8, 3.6))
plt.xlabel("% de estudiantes sin dato"); plt.title("Faltantes por variable"); plt.show()

incompleta = X.isna().any(axis=1)
comparacion = pd.DataFrame({
    "estudiantes": incompleta.value_counts(),
    "% en el cuarto más bajo": (y.groupby(incompleta).mean() * 100).round(1),
}).rename(index={False: "fila completa", True: "con algún faltante"})
display(comparacion)
print(f"dropna() borraría el {incompleta.mean() * 100:.1f} % de las filas.")

Los dos porcentajes de la tabla no coinciden. Quien deja preguntas sin responder queda en el cuarto más bajo con más frecuencia que quien responde todo. `dropna()` sacaría de la tabla, con preferencia, a los estudiantes que la Secretaría está buscando, y lo haría sin avisar. Eso es un **sesgo de selección** creado por la limpieza.

**Decisión:** no se borra ninguna fila por faltantes. Cada faltante se reemplaza por la mediana de su variable, calculada solo con los estudiantes de entrenamiento. Además, la educación del padre trae muchos «No sabe» y «No Aplica», y eso también es información sobre el hogar.

**Quinta trampa: columnas que ya conocen la respuesta.** La tabla trae columnas que se calcularon después del examen. Con ellas cualquier modelo acierta, pero ninguna existe el día en que se reparten los cupos.

In [ ]:
filtran = ["punt_lectura_critica", "punt_matematicas", "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles",
           "punt_global", "percentil_global", "desemp_matematicas", "estu_generacion_e"]
print(pd.crosstab(datos["estu_generacion_e"], y.map({0: "no quedó abajo", 1: "quedó abajo"})).to_string())

Los puntajes por área y el nivel de desempeño son pedazos de la respuesta. `estu_generacion_e` es más sutil. Marca a quién se le reconoció después un beneficio del programa Generación E, y las dos categorías de «Excelencia» no tienen a nadie en el cuarto más bajo: el beneficio se otorgó mirando el puntaje. En la sección 3.5 medimos cuánto cambia el resultado con cada una de estas columnas.

Hay otras dos columnas que dejo por fuera por una razón distinta. `estu_inse_individual` y `estu_nse_individual` son el índice socioeconómico que el ICFES calcula con el cuestionario. No filtran la respuesta, pero el ICFES las publica con los resultados y no tengo cómo saber si la Secretaría las tendría a tiempo. Uso las respuestas originales del cuestionario, y en la sección 2.3 comparo con el índice.

### 2.2 Conocer el dato antes de modelarlo

Cuatro miradas: cómo se reparte el puntaje, cómo cambia la tasa de puntaje bajo entre grupos, quiénes presentan el examen y quiénes no están en la tabla. Para un grupo $g$, la tasa es

$$\hat p_g=\frac{1}{n_g}\sum_{i\in g}y_i,\qquad \text{con error estándar}\quad \sqrt{\frac{\hat p_g(1-\hat p_g)}{n_g}}.$$

El error estándar importa porque algunos grupos son pequeños. Una tasa calculada con cuarenta estudiantes se mueve mucho más que una calculada con cuatro mil.

In [ ]:
corte = datos.loc[y == 1, "punt_global"].max()
fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 3.8))
izq.hist(datos["punt_global"], bins=60, color="#16302B")
izq.axvline(corte, color="#A8432C", lw=2); izq.text(corte - 8, izq.get_ylim()[1] * 0.85, f"percentil 25 nacional\n({corte} puntos)", color="#A8432C", fontsize=9, ha="right")
izq.set(title=f"Puntaje global, {DEPARTAMENTO.title()}", xlabel="puntos (0 a 500)", ylabel="estudiantes")

colegios = pd.DataFrame({"n": y.groupby(colegio).size(), "tasa": y.groupby(colegio).mean() * 100})
der.scatter(colegios["n"], colegios["tasa"], s=14, color="#3E7CB1", alpha=0.7)
der.axhline(y.mean() * 100, color="gray", ls=":")
der.set(title="Cada punto es un colegio", xlabel="estudiantes que presentaron", ylabel="% en el cuarto más bajo")
plt.tight_layout(); plt.show()
print(f"Colegios: {len(colegios)} | mediana de estudiantes por colegio: {colegios['n'].median():.0f}")
print(f"Colegios sin ningún estudiante abajo: {(colegios['tasa'] == 0).sum()} | con más de la mitad abajo: {(colegios['tasa'] > 50).sum()}")

La gráfica de la derecha anticipa una decisión de la sección 3. Los estudiantes de un mismo colegio se parecen entre sí: hay colegios donde casi nadie queda abajo y colegios donde queda más de la mitad. Los colegios pequeños ocupan los dos extremos, que es lo que predice la fórmula del error estándar.

In [ ]:
def tasa_por(grupo, orden=None):
    tabla = pd.DataFrame({"n": y.groupby(grupo).size(), "tasa": y.groupby(grupo).mean() * 100})
    tabla["ee"] = np.sqrt(tabla["tasa"] * (100 - tabla["tasa"]) / tabla["n"])
    return tabla.reindex(orden) if orden else tabla

paneles = [
    ("Educación de la madre", tasa_por(datos["fami_educacionmadre"].fillna("(sin respuesta)"), list(niveles_educacion) + ["No sabe", "(sin respuesta)"])),
    ("Estrato de la vivienda", tasa_por(datos["fami_estratovivienda"].fillna("(sin respuesta)"), [f"Estrato {e}" for e in range(1, 7)] + ["Sin Estrato", "(sin respuesta)"])),
    ("Jornada del colegio", tasa_por(datos["cole_jornada"]).sort_values("tasa")),
    ("Naturaleza y zona", tasa_por(datos["cole_naturaleza"].str.capitalize() + ", " + datos["cole_area_ubicacion"].str.lower()).sort_values("tasa")),
]
fig, ejes = plt.subplots(2, 2, figsize=(13, 8))
for eje, (titulo, tabla) in zip(ejes.ravel(), paneles):
    tabla = tabla.dropna()
    eje.barh([f"{nombre[:34]} (n={int(n)})" for nombre, n in zip(tabla.index, tabla["n"])], tabla["tasa"], xerr=2 * tabla["ee"], color="#E0A526", ecolor="#16302B")
    eje.axvline(y.mean() * 100, color="gray", ls=":"); eje.invert_yaxis(); eje.tick_params(axis="y", labelsize=8)
    eje.set(title=titulo, xlabel="% en el cuarto más bajo")
plt.tight_layout(); plt.show()

Tres lecturas. La primera es que la tasa baja de forma ordenada con la educación de la madre. La segunda es que el estrato se comporta peor de lo que uno espera en la parte alta: los estratos 5 y 6 tienen pocos estudiantes y barras de error anchas, así que su tasa dice poco. La tercera es que las jornadas nocturna y sabatina están muy por encima del resto. En esas jornadas estudian, sobre todo, adultos que retomaron el bachillerato.

En las dos gráficas de arriba, la barra de «sin respuesta» queda a la derecha de la línea punteada, que es la tasa del departamento. Es la cuarta trampa otra vez.

**¿Quiénes están en la tabla y quiénes no?** Este es el sesgo que ninguna limpieza arregla.

In [ ]:
print(f"Mujeres entre quienes presentaron: {X['mujer'].mean() * 100:.1f} %")
print(f"Mayores de 20 años: {(X['edad'] > 20).mean() * 100:.1f} % | en jornada nocturna o sabatina: {X['nocturna_o_sabatina'].mean() * 100:.1f} %")
print(f"Trabajan más de 20 horas a la semana: {(X['horas_trabajo'] >= 3).mean() * 100:.1f} %")
resumen_sexo = pd.DataFrame({"n": y.groupby(datos["estu_genero"]).size(),
                             "puntaje medio": datos.groupby("estu_genero")["punt_global"].mean().round(1),
                             "% en el cuarto más bajo": (y.groupby(datos["estu_genero"]).mean() * 100).round(1)})
display(resumen_sexo)

La tabla solo contiene a quienes llegaron a presentar el examen. Los jóvenes que dejaron el colegio antes de once no aparecen, y son los que más habrían necesitado el refuerzo. Por eso las mujeres son más de la mitad de la tabla: ese número habla de quién llega a once, y este dato no alcanza para decir por qué.

De ahí sale una advertencia para leer la tabla de arriba. La diferencia entre hombres y mujeres se calcula sobre dos grupos que pasaron filtros distintos antes de llegar al examen. Sirve para describir a quienes presentaron. No sirve para comparar a todos los jóvenes de la misma edad.

### 2.3 PCA: ¿cuántas dimensiones tiene «el nivel socioeconómico»?

Nueve variables describen el hogar: estrato, educación de la madre y del padre, internet, computador, automóvil, libros, cuartos y personas. Si todas midieran lo mismo, bastaría un solo número. Tomo la matriz $Z$ de esas variables estandarizadas y descompongo su matriz de correlaciones:

$$R=\frac{1}{n-1}Z^\top Z=V\Lambda V^\top,\qquad \text{varianza explicada por el componente } j=\frac{\lambda_j}{\sum_k\lambda_k}.$$

El primer componente, $z_i^\top v_1$, es la combinación de las nueve variables que más varía entre estudiantes. El ICFES publica su propio índice socioeconómico, el INSE. Si el primer componente y el INSE cuentan la misma historia, su correlación será alta.

**Predicción escrita:** ¿cuánta varianza explicará el primer componente? Con nueve variables independientes sería 11 %. **Mi apuesta:**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

hogar = ["estrato", "educacion_madre", "educacion_padre", "internet", "computador", "automovil", "libros", "cuartos", "personas_hogar"]
Z = StandardScaler().fit_transform(X[hogar].fillna(X[hogar].median()))     # mediana de todos: aquí solo se describe
pca = PCA().fit(Z)
explicada = pca.explained_variance_ratio_ * 100
cargas = pd.DataFrame(pca.components_.T[:, :2], index=hogar, columns=["CP1", "CP2"])
if cargas["CP1"].sum() < 0:
    cargas["CP1"] = -cargas["CP1"]                                          # el signo de un componente es arbitrario
cp1 = pd.Series(Z @ cargas["CP1"].values, index=X.index)

fig, (izq, centro, der) = plt.subplots(1, 3, figsize=(14, 3.9))
izq.bar(range(1, 10), explicada, color="#16302B"); izq.axhline(100 / 9, color="gray", ls=":")
izq.set(title="Varianza explicada por componente", xlabel="componente", ylabel="%")
cargas["CP1"].sort_values().plot.barh(ax=centro, color="#E0A526"); centro.set(title="Cargas del primer componente")
quintil = pd.qcut(cp1, 5, labels=["1 (más bajo)", "2", "3", "4", "5 (más alto)"])
(y.groupby(quintil, observed=True).mean() * 100).plot.bar(ax=der, color="#A8432C", rot=0)
der.set(title="Tasa de puntaje bajo por quintil del CP1", xlabel="quintil del primer componente", ylabel="%")
plt.tight_layout(); plt.show()

inse = datos["estu_inse_individual"]
print(f"El primer componente explica {explicada[0]:.0f} % de la varianza; el segundo, {explicada[1]:.0f} %.")
print(f"Correlación entre el primer componente y el INSE del ICFES: {cp1.corr(inse):.2f}")

El primer componente carga con el mismo signo sobre casi todas las variables y explica varias veces lo que explicaría una variable suelta. Se puede leer como un índice de recursos del hogar, y su correlación con el INSE dice qué tan cerca está del índice oficial. La gráfica de la derecha muestra que ese solo número ya ordena a los estudiantes por riesgo.

El resto de la varianza no desaparece. Tener automóvil y tener libros no son la misma cosa, y por eso el modelo recibirá las variables originales y no el componente. El PCA aquí sirve para describir.

## 3. Modelado

### 3.1 La métrica y la trampa de la exactitud

Un modelo de clasificación entrega para cada estudiante una probabilidad estimada $\hat p_i$ de quedar abajo. Para medirlo hay que cruzar lo que predijo con lo que pasó. Con un umbral de 0,5 salen cuatro casillas: verdaderos positivos (VP), falsos positivos (FP), verdaderos negativos (VN) y falsos negativos (FN). Las tres medidas más usadas son

$$\text{exactitud}=\frac{VP+VN}{n},\qquad \text{sensibilidad}=\frac{VP}{VP+FN},\qquad \text{precisión}=\frac{VP}{VP+FP}.$$

La exactitud es la que reporta todo el mundo. Veamos qué dice aquí, con el modelo más sencillo y con un «modelo» que no hace nada.

Primero la partición. Todos los modelos de este cuaderno se entrenan con unos colegios, se escogen con otros y se reportan con otros:

$$\underbrace{60\,\%\ \text{de los colegios}}_{\text{entrenamiento}}\ \Big|\ \underbrace{20\,\%}_{\text{validación: se escoge}}\ \Big|\ \underbrace{20\,\%}_{\text{prueba: se reporta}}$$

Se parte por colegio y no por estudiante por lo que vimos en la sección 2.2. Si los compañeros de un mismo salón quedan repartidos entre entrenamiento y prueba, el examen del modelo se parece al entrenamiento más de lo que se parecerá un colegio nuevo.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

resto, prueba = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=0).split(X, y, colegio))
entrena, valida = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(X.iloc[resto], y.iloc[resto], colegio.iloc[resto]))
E, V, P = (np.zeros(len(X), dtype=bool) for _ in range(3))
E[resto[entrena]] = True; V[resto[valida]] = True; P[prueba] = True

for nombre, tramo in [("Entrenamiento", E), ("Validación", V), ("Prueba", P)]:
    print(f"{nombre:14s} {tramo.sum():5d} estudiantes | {colegio[tramo].nunique():3d} colegios | {y[tramo].mean() * 100:.1f} % abajo")

In [ ]:
logistica = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[E], y[E])
prob = logistica.predict_proba(X[V])[:, 1]
dice_bajo = prob >= 0.5
real = y[V].values

def tres_medidas(real, dice):
    vp, fp, fn = ((dice == 1) & (real == 1)).sum(), ((dice == 1) & (real == 0)).sum(), ((dice == 0) & (real == 1)).sum()
    return {"exactitud": (dice == real).mean(), "sensibilidad": vp / (vp + fn), "precisión": vp / (vp + fp) if vp + fp else np.nan}

display(pd.DataFrame({"decir que nadie queda abajo": tres_medidas(real, np.zeros(len(real))),
                      "regresión logística, umbral 0,5": tres_medidas(real, dice_bajo)}).T.round(3))
print(f"El modelo señala a {dice_bajo.sum()} de {len(real)} estudiantes. Quedaron abajo {real.sum()}.")

Decir que nadie queda abajo acierta con casi todos los estudiantes, porque la mayoría no queda abajo. El modelo apenas mejora esa exactitud, y su sensibilidad muestra que deja pasar a la mayoría de quienes la Secretaría busca. La exactitud premia acertar en el grupo grande y no se entera de lo que pasa en el pequeño.

El umbral de 0,5 tampoco tiene nada que ver con el problema. La Secretaría tiene un número de cupos, así que lo que necesita del modelo es un **orden**. Por eso medimos con la captación de la sección 1 y, como segunda medida, con el área bajo la curva ROC, que resume la calidad del orden sin fijar un cupo:

$$\text{AUC}=P\bigl(\hat p_i>\hat p_j\ \big|\ y_i=1,\ y_j=0\bigr).$$

Un AUC de 0,5 es ordenar al azar y un AUC de 1 es ordenar perfecto.

### 3.2 La línea base

Dos reglas que no necesitan modelo. La primera reparte los cupos por sorteo, y su captación esperada es el mismo cupo. La segunda ordena a los estudiantes por la educación de la madre, de menor a mayor, que es lo que haría alguien con sentido común y una sola columna.

In [ ]:
def captacion(real, puntaje, cupo=CUPO):
    """Fracción de los que quedaron abajo que cae entre los primeros `cupo` del orden. Los empates se rompen por sorteo."""
    real, puntaje = np.asarray(real), np.asarray(puntaje, dtype=float)
    k = int(round(cupo * len(real)))
    orden = np.lexsort((np.random.default_rng(0).random(len(real)), -puntaje))
    return float(real[orden[:k]].sum() / real.sum())

mediana_madre = X.loc[E, "educacion_madre"].median()
reglas = {
    "sorteo": pd.Series(np.random.default_rng(1).random(len(X)), index=X.index),
    "regla: educación de la madre": -X["educacion_madre"].fillna(mediana_madre),
}
errores = {nombre: {"captación en validación": captacion(y[V], regla[V]), "captación en prueba": captacion(y[P], regla[P]),
                    "AUC en prueba": roc_auc_score(y[P], regla[P])} for nombre, regla in reglas.items()}
display(pd.DataFrame(errores).T.round(3))
print(f"Techo de la captación en prueba: {min(1, CUPO / y[P].mean()):.2f}")

La última línea trae un detalle que se olvida con facilidad. La captación tiene un techo, que depende de cuántos estudiantes quedan abajo:

$$\text{captación máxima}=\min\!\left(1,\ \frac{\text{cupo}}{\hat p}\right).$$

Si quedan abajo más estudiantes que cupos, ni un oráculo los cubre a todos. Guarden esa idea para la sección 4.5.

### 3.3 Cinco modelos y un protocolo para escoger

| Modelo | Idea en una línea |
|---|---|
| Regresión logística | $\log\dfrac{p_i}{1-p_i}=\beta_0+x_i^\top\beta$: cada variable suma o resta al logaritmo de la razón de probabilidades |
| $k$ vecinos | La fracción que quedó abajo entre los $k$ estudiantes de entrenamiento más parecidos |
| Árbol de decisión | Preguntas de sí o no, una tras otra, que van separando grupos con tasas distintas |
| Bosque aleatorio | El promedio de muchos árboles entrenados sobre remuestreos del dato |
| *Gradient boosting* | Una suma de árboles pequeños, cada uno corrigiendo el error de los anteriores |

La regla queda fijada antes de mirar: gana quien tenga mayor captación en **validación**, y su captación en **prueba** es la que se le reporta a la Secretaría. Los faltantes se llenan con la mediana de entrenamiento dentro de cada modelo, para que la validación y la prueba no participen en ese cálculo.

In [ ]:
def cinco_modelos():
    llenar = lambda: SimpleImputer(strategy="median")
    return {
        "logística": make_pipeline(llenar(), StandardScaler(), LogisticRegression(max_iter=1000)),
        "k vecinos": make_pipeline(llenar(), StandardScaler(), KNeighborsClassifier(n_neighbors=50)),
        "árbol": make_pipeline(llenar(), DecisionTreeClassifier(max_depth=5, random_state=0)),
        "bosque": make_pipeline(llenar(), RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1)),
        "boosting": make_pipeline(llenar(), HistGradientBoostingClassifier(random_state=0)),
    }

puntajes = {nombre: regla[P] for nombre, regla in reglas.items()}        # el orden que cada candidato propone en la prueba
for nombre, modelo in cinco_modelos().items():
    modelo.fit(X[E], y[E])                                                # se entrena con los colegios de entrenamiento
    en_validacion = captacion(y[V], modelo.predict_proba(X[V])[:, 1])
    modelo.fit(X[E | V], y[E | V])                                        # se reentrena con todo lo que no es prueba
    puntajes[nombre] = pd.Series(modelo.predict_proba(X[P])[:, 1], index=X.index[P])
    errores[nombre] = {"captación en validación": en_validacion, "captación en prueba": captacion(y[P], puntajes[nombre]),
                       "AUC en prueba": roc_auc_score(y[P], puntajes[nombre])}
todos = pd.DataFrame(errores).T

todos[["captación en validación", "captación en prueba"]].sort_values("captación en validación").plot.barh(figsize=(9, 4), color=["#16302B", "#E0A526"])
plt.axvline(CUPO, color="gray", ls=":"); plt.xlabel("captación"); plt.title("Siete candidatos: validación para escoger, prueba para reportar"); plt.show()
display(todos.sort_values("captación en validación", ascending=False).round(3))

In [ ]:
fracciones = np.linspace(0.02, 1, 50)
plt.figure(figsize=(8, 4.2))
for nombre, puntaje in puntajes.items():
    estilo = {"color": "gray", "ls": ":"} if nombre == "sorteo" else {"color": "#E0A526", "lw": 2} if nombre.startswith("regla") else {"lw": 1.2}
    plt.plot(fracciones * 100, [captacion(y[P], puntaje, f) for f in fracciones], label=nombre, **estilo)
plt.axvline(CUPO * 100, color="#A8432C", lw=1); plt.text(CUPO * 100 + 1, 0.05, "cupo", color="#A8432C")
plt.xlabel("% de estudiantes con cupo"); plt.ylabel("captación"); plt.title("Captación según el tamaño del programa, en la prueba"); plt.legend(fontsize=8); plt.show()

La segunda gráfica muestra la captación para cualquier tamaño de programa, y la línea vertical marca el cupo de la Secretaría. Mientras más arriba queda una curva sobre la diagonal del sorteo, mejor ordena ese candidato. Fíjense en qué tan juntas van las curvas de los cinco modelos y en qué tan lejos queda la regla de una sola columna.

### 3.4 El veredicto

In [ ]:
mejor = todos.drop(index="sorteo")["captación en validación"].idxmax()
resumen = todos["captación en prueba"].round(3)
regla = "regla: educación de la madre"
print(f"Ganador en validación: {mejor}")
print(f"Captación en prueba: {resumen[mejor]:.3f} | regla de una columna: {resumen[regla]:.3f} | sorteo: {resumen['sorteo']:.3f} | techo: {min(1, CUPO / y[P].mean()):.2f}")

Lean los tres números juntos. El sorteo dice cuánto se logra sin saber nada. La regla dice cuánto se logra con una columna y sin modelo. La distancia entre la regla y el ganador es lo que de verdad aporta el aprendizaje automático en este problema, y la distancia entre el ganador y el techo es lo que el cuestionario no alcanza a ver.

### 3.5 La trampa de la clase: columnas que filtran la respuesta

En la sección 2.1 separé nueve columnas que se calcularon después del examen. Ahora medimos qué pasa si alguien las deja en la tabla por descuido. El modelo es el mismo y la partición es la misma. Solo cambian las columnas.

**Predicción escrita:** ¿cuánto subirá la captación si el modelo recibe el nivel de desempeño en matemáticas? **Mi apuesta:**

In [ ]:
extras = pd.DataFrame({
    "generacion_e": (datos["estu_generacion_e"] != "NO").astype(float),
    "generacion_e_excelencia": datos["estu_generacion_e"].str.contains("EXCELENCIA").astype(float),
    "desemp_matematicas": datos["desemp_matematicas"].astype(float),
    "punt_ingles": datos["punt_ingles"].astype(float),
    "punt_lectura_critica": datos["punt_lectura_critica"].astype(float),
})
escenarios = {
    "solo lo que se sabe en la inscripción": [],
    "+ Generación E": ["generacion_e", "generacion_e_excelencia"],
    "+ desempeño en matemáticas": ["desemp_matematicas"],
    "+ dos puntajes por área": ["punt_ingles", "punt_lectura_critica"],
}
fuga = {}
for nombre, columnas in escenarios.items():
    X_con = pd.concat([X, extras[columnas]], axis=1)
    modelo = cinco_modelos()["boosting"].fit(X_con[E | V], y[E | V])
    prob = modelo.predict_proba(X_con[P])[:, 1]
    fuga[nombre] = {"captación en prueba": captacion(y[P], prob), "AUC en prueba": roc_auc_score(y[P], prob)}
display(pd.DataFrame(fuga).T.round(3))

La primera fila es el resultado honesto. Las demás son resultados que no se pueden cumplir, porque usan información que no existe cuando se reparten los cupos. A esto se le llama **fuga de información**. Lo peligroso es que nada falla. El código corre, la métrica sube y el informe se ve mejor.

La fila de Generación E enseña otra cosa. Esa columna casi no mueve la captación, porque delata a los estudiantes de arriba y la Secretaría busca a los de abajo. Una columna puede filtrar la respuesta y no notarse en la métrica que uno escogió. Por eso las columnas con fuga se sacan por lo que son, y no por lo que le hacen al número.

La fuga no siempre es tan visible como un puntaje. La pregunta que la detecta es siempre la misma: el día en que se toma la decisión, ¿esta columna ya existe?

### 3.6 Explicar: ¿qué acompaña al puntaje bajo?

La regresión logística permite leer cada variable. Como las variables entraron estandarizadas, $e^{\beta_j}$ es la **razón de probabilidades** asociada a subir una desviación estándar en la variable $j$, dejando quietas las demás:

$$\text{OR}_j=e^{\beta_j}=\frac{p/(1-p)\ \text{con}\ x_j+1\ \text{d.e.}}{p/(1-p)\ \text{con}\ x_j}.$$

Un valor de 1,3 dice que la razón entre quedar abajo y no quedar abajo sube 30 %. Un valor de 0,8 dice que baja 20 %.

In [ ]:
logistica = cinco_modelos()["logística"].fit(X[E | V], y[E | V])
razones = pd.Series(np.exp(logistica[-1].coef_[0]), index=variables).sort_values()
razones.plot.barh(figsize=(8, 5.5), color=["#3E7CB1" if r < 1 else "#A8432C" for r in razones])
plt.axvline(1, color="gray"); plt.xlabel("razón de probabilidades por desviación estándar"); plt.title("Qué acompaña a quedar en el cuarto más bajo"); plt.show()
print(razones.round(2).to_string())

Comparen con su cuarta apuesta. Antes de sacar conclusiones, tres advertencias.

La primera es que estos números describen asociaciones entre estudiantes que presentaron el examen. Que los libros en la casa acompañen un mejor resultado no significa que regalar libros lo produzca.

La segunda es que las variables se pisan. Estrato, computador e internet miden cosas parecidas, como mostró el PCA, y la regresión reparte entre ellas un efecto que comparten. Un coeficiente pequeño en el estrato no dice que el estrato no importe.

La tercera es que hay variables, como la edad y la jornada, que cuentan una historia de trayectoria escolar. Un estudiante de 25 años en jornada sabatina no llegó ahí por su edad.

### 3.7 Sesgos: ¿a quién le acierta y a quién le falla el modelo?

Un modelo puede tener buena captación en promedio y repartir sus errores de forma desigual. Para cada grupo miro cuatro números en la prueba: qué fracción quedó abajo, qué fracción recibe cupo, la captación dentro del grupo y la fracción de **cupos de sobra**, que son los estudiantes que no quedaron abajo y recibieron cupo:

$$\text{cupos de sobra}_g=\frac{\#\{i\in g:\ y_i=0\ \text{y recibe cupo}\}}{\#\{i\in g:\ y_i=0\}}.$$

In [ ]:
elegido = mejor if resumen[mejor] > resumen[regla] else regla          # el más simple entre los que no pierden
puntaje = puntajes[elegido]
k = int(round(CUPO * P.sum()))
orden = np.lexsort((np.random.default_rng(0).random(P.sum()), -puntaje.values))
cupo = pd.Series(False, index=X.index[P]); cupo.iloc[orden[:k]] = True

def auditar(grupo):
    partes = {}
    for valor, indices in grupo[P].groupby(grupo[P]).groups.items():
        yy, cc = y[indices], cupo[indices]
        partes[valor] = {"n": len(yy), "% que quedó abajo": yy.mean() * 100, "% con cupo": cc.mean() * 100,
                         "captación": cc[yy == 1].mean(), "cupos de sobra": cc[yy == 0].mean()}
    return pd.DataFrame(partes).T

grupos = {
    "sexo": datos["estu_genero"].map({"F": "mujeres", "M": "hombres"}),
    "zona": datos["cole_area_ubicacion"].str.lower(),
    "naturaleza": datos["cole_naturaleza"].str.lower(),
    "jornada": X["nocturna_o_sabatina"].map({1.0: "nocturna o sabatina", 0.0: "diurna"}),
}
auditoria = pd.concat({nombre: auditar(grupo) for nombre, grupo in grupos.items()})
print("Candidato auditado:", elegido)
display(auditoria.round({"n": 0, "% que quedó abajo": 1, "% con cupo": 1, "captación": 2, "cupos de sobra": 2}))

La tabla se lee por parejas. Cuando dos grupos tienen tasas reales parecidas y el modelo le da muchos más cupos a uno, hay algo que revisar. Cuando un grupo tiene una tasa real mucho más alta, es razonable que reciba más cupos, y aun así conviene mirar su columna de cupos de sobra: ahí se ve si el modelo aprendió a señalar al grupo entero en vez de distinguir a las personas dentro de él.

Miren primero el sexo y después la jornada. En el grupo donde la captación es más baja están los estudiantes que quedaron abajo sin que el modelo los viera. Esa es la gente que la decisión deja por fuera, y la sección 4 tiene que decirlo.

Esta auditoría tiene un límite. Solo puedo auditar por las variables que el dato trae.

## 4. Comunicación

### 4.1 Decidir: ¿a quiénes se les ofrece el cupo?

La captación de la prueba salió de unos pocos colegios. Con otros colegios habría salido otro número. Para ponerle un margen uso *bootstrap* por colegio: saco muchas veces, con reposición, una muestra de los colegios de prueba y recalculo la captación. El intervalo va del percentil 5 al 95 de esos resultados,

$$\bigl[\hat Q_{0{,}05}(\text{captación}^*),\ \hat Q_{0{,}95}(\text{captación}^*)\bigr].$$

Se remuestrean colegios y no estudiantes por la misma razón por la que se partió por colegio.

In [ ]:
colegios_prueba = colegio[P].unique()
filas_por_colegio = {c: np.flatnonzero(colegio[P].values == c) for c in colegios_prueba}
sorteo = np.random.default_rng(0)
replicas = {"elegido": [], "regla": []}
for _ in range(500):
    filas = np.concatenate([filas_por_colegio[c] for c in sorteo.choice(colegios_prueba, len(colegios_prueba))])
    replicas["elegido"].append(captacion(y[P].values[filas], puntajes[elegido].values[filas]))
    replicas["regla"].append(captacion(y[P].values[filas], puntajes[regla].values[filas]))
bajo_, alto_ = np.percentile(replicas["elegido"], [5, 95])
ventaja = np.array(replicas["elegido"]) - np.array(replicas["regla"])

precision = y[P][cupo].mean()
print(f"Candidato elegido: {elegido}")
print(f"Captación en prueba: {resumen[elegido]:.2f}, con intervalo de 90 % entre {bajo_:.2f} y {alto_:.2f}")
print(f"Ventaja sobre la regla de una columna: {np.median(ventaja):+.2f} (intervalo de 90 %: {np.percentile(ventaja, 5):+.2f} a {np.percentile(ventaja, 95):+.2f})")
print(f"De cada 100 cupos, {precision * 100:.0f} llegan a estudiantes que quedaron en el cuarto más bajo. Por sorteo llegarían {y[P].mean() * 100:.0f}.")
print(f"De cada 100 estudiantes que quedaron abajo, {(1 - resumen[elegido]) * 100:.0f} se quedan sin cupo.")

Los últimos dos renglones son los que entiende quien no es técnico. El primero dice qué tan bien gastados quedan los cupos. El segundo dice cuánta gente se queda por fuera, y esa cifra hay que entregarla siempre junto con la primera.

### La decisión en tres frases

Escríbanla para una secretaria de educación que no sabe qué es una regresión logística. Una frase para lo que recomiendan, una para la evidencia y una para lo que puede salir mal.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| «Bajo» es percentil nacional de 25 o menos | Un puntaje fijo; el cuartil del departamento | El percentil lo publica el ICFES y permite comparar departamentos |
| Edad por fuera de 13 a 70 años se declara faltante | Borrar la fila; dejar el valor | La fecha es un error de digitación, el estudiante sí presentó |
| Puntaje global igual a cero sale de la tabla | Tratarlo como el puntaje más bajo | No hay una prueba respondida que predecir |
| «No sabe» y «No Aplica» se tratan como faltantes; «Sin Estrato» tiene su columna | Ponerles un número dentro del orden | No son un nivel más bajo ni más alto |
| Los faltantes se llenan con la mediana de entrenamiento | `dropna()` | Quien no responde queda abajo con más frecuencia: borrar sesga |
| Fuera las nueve columnas calculadas después del examen, y el INSE | Dejarlas | No existen el día en que se reparten los cupos |
| Partición y *bootstrap* por colegio | Por estudiante | Los compañeros de colegio se parecen |
| Captación con cupo de 25 % como métrica | Exactitud | La Secretaría reparte un número fijo de cupos |

### 4.3 Riesgos, ética y mantenimiento

Este dato es de personas, casi todas menores de edad cuando presentaron el examen. Respondan en dos o tres líneas cada una.

- El modelo se entrenó con la cohorte de 2020 y se usaría con una cohorte posterior. ¿Qué habría que comprobar antes de confiar en él?
- La lista de cupos sale de variables como la educación de los padres y el estrato. ¿Qué pasa si un estudiante o un colegio se entera de qué respuestas dan cupo?
- Ofrecer refuerzo no le hace daño a nadie. ¿Cambiaría su opinión sobre este mismo modelo si se usara para negar una beca?
- Según la auditoría de la sección 3.7, ¿qué grupo queda peor atendido? ¿Qué le propondrían a la Secretaría para ese grupo?
- Los jóvenes que dejaron el colegio antes de once no están en la tabla. ¿Qué decisión de política quedaría mal informada si alguien olvida eso?

### 4.4 Procedencia

In [ ]:
procedencia()["saber11"]

### 4.5 Cambien una línea

Vuelvan a la celda donde dice `DEPARTAMENTO = "CALDAS"`, pongan `DEPARTAMENTO = "CHOCO"` y corran todo de nuevo con *Entorno de ejecución*, *Ejecutar todo*. El ICFES escribe los departamentos en mayúsculas y casi siempre sin tilde. Pueden escoger otro departamento si lo prefieren. Llenen la tabla con sus dos corridas.

| | Caldas | Su departamento |
|---|---|---|
| Estudiantes y colegios | | |
| % en el cuarto más bajo del país | | |
| Techo de la captación | | |
| Ganador en validación | | |
| Captación del ganador en prueba | | |
| Captación de la regla de una columna | | |
| De cada 100 cupos, cuántos llegan a quien quedó abajo | | |

Con la tabla llena, respondan en tres líneas. El corte del «cuarto más bajo» es nacional y el cupo es de 25 % en los dos departamentos. ¿Qué le pasa a la métrica cuando la tasa del departamento se aleja de 25 %? ¿Qué cambiarían en la formulación, el corte o el cupo, y por qué? **Mi respuesta:**